### Silver Data Transformation

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_titles = spark.read.format("delta").option("inferSchema", True).option("header",True).load("abfss://bronze@netflixstorageaccount.dfs.core.windows.net/netflix_titles")

In [0]:
display(df_titles)

In [0]:
df_titles = df_titles.fillna({"duration_minutes": "0","duration_seasons": "1"})

In [0]:
display(df_titles)

In [0]:
df_titles = df_titles.withColumn("duration_seasons", col("duration_seasons").try_cast("int"))\
                    .withColumn("duration_minutes", col("duration_minutes").try_cast("int"))

In [0]:
df_titles.display()

In [0]:
df_titles = df_titles.withColumn("Shorttitle", split(col("title"),":")[0])

In [0]:
display(df_titles)

In [0]:
df_titles = df_titles.withColumn("rating",split(col("rating"),"-")[0])

In [0]:
display(df_titles)

In [0]:
df_titles = df_titles.withColumn("Type_Flag", when(col("type")=="Movie",1)
                    .when(col("type")=="TV Show",2)
                    .otherwise(0))
display(df_titles)

In [0]:
from pyspark.sql.window import Window

In [0]:
df_titles = df_titles.withColumn("Duration_Ranking",dense_rank().over(Window.orderBy(col("duration_minutes").desc())))

In [0]:
display(df_titles)

In [0]:
df_titles = df_titles.groupBy("type").agg(count("*").alias("Total_Count"))
display(df_titles)

Databricks visualization. Run in Databricks to view.

In [0]:
df_titles.write.format("delta").mode("overwrite").option("path","abfss://silver@netflixstorageaccount.dfs.core.windows.net/netflix_titles").save()